# Lab 01: A simple M/M/1 queue simulation

**Problem Statement:**
Write a program which performs a simple M/M/1 queue simulation. This program requires parameters for Mean Inter Arrival time of customers, Mean Service time as well as maximum number of customers. The simulation is started with a single-server queue with a FIFO queuing discipline. For M/M/1 queue, the customer inter-arrival time and the service time are both exponentially distributed. This simulation shows Average delay in queue, Average number in queue, Server utilization, and Time simulation ended.


### Theory and Formulas (Book Matching)

In an **M/M/1 queue**:
- **M**: Memoryless (Exponential) inter-arrival times with rate $\lambda$ (Mean Inter-Arrival Time = $1/\lambda$).
- **M**: Memoryless (Exponential) service times with rate $\mu$ (Mean Service Time = $1/\mu$).
- **1**: Single server.

**Key Metrics:**
1. **Server Utilization ($\rho$)**: The fraction of time the server is busy. Expected $\rho = \lambda / \mu$.
2. **Average Delay in Queue**: The average time a customer spends waiting in the queue before service begins.
3. **Average Number in Queue**: The average length of the queue over time.

We will simulate this system using a **Discrete Event Simulation (DES)** approach.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. Initialization and Parameters
# ==========================================
np.random.seed(42) # For reproducible results

MEAN_INTERARRIVAL = 2.0  # Mean time between arrivals (1/lambda)
MEAN_SERVICE = 1.5       # Mean time to serve a customer (1/mu)
MAX_CUSTOMERS = 1000     # Maximum number of customers to simulate

# Simulation State Variables
clock = 0.0              # Current simulation time
server_busy = False      # Server state (True if busy, False if idle)
queue = []               # Queue holding arrival times of waiting customers

# Statistical Counters
total_delay = 0.0        # Sum of delays of all customers
area_queue = 0.0         # Integral of queue length over time
area_server = 0.0        # Integral of server busy time
customers_delayed = 0    # Number of customers who have completed their delay

# Event List
# We use exponential distribution for M/M/1 queue
next_arrival = np.random.exponential(MEAN_INTERARRIVAL)
next_departure = float('inf') # No departures initially

# Data collection for plotting
time_points = [0.0]
queue_lengths = [0]

# ==========================================
# 2. Simulation Loop
# ==========================================
while customers_delayed < MAX_CUSTOMERS:
    # Determine the next event (Arrival or Departure)
    next_event_time = min(next_arrival, next_departure)
    
    # Update area statistics before advancing the clock
    time_since_last_event = next_event_time - clock
    area_queue += len(queue) * time_since_last_event
    area_server += (1 if server_busy else 0) * time_since_last_event
    
    # Advance clock to next event time
    clock = next_event_time
    
    # --- PROCESS ARRIVAL EVENT ---
    if next_arrival <= next_departure:
        if server_busy:
            # Server is busy, customer joins the queue
            queue.append(clock)
        else:
            # Server is idle, customer starts service immediately (delay = 0)
            server_busy = True
            next_departure = clock + np.random.exponential(MEAN_SERVICE)
        
        # Schedule the next arrival
        next_arrival = clock + np.random.exponential(MEAN_INTERARRIVAL)
        
    # --- PROCESS DEPARTURE EVENT ---
    else:
        customers_delayed += 1
        
        if len(queue) > 0:
            # Queue is not empty, next customer starts service
            arrival_time = queue.pop(0)
            delay = clock - arrival_time
            total_delay += delay
            
            # Schedule this customer's departure
            next_departure = clock + np.random.exponential(MEAN_SERVICE)
        else:
            # Queue is empty, server becomes idle
            server_busy = False
            next_departure = float('inf')
            
    # Record data for visualization
    time_points.append(clock)
    queue_lengths.append(len(queue))

# ==========================================
# 3. Calculate and Print Performance Measures
# ==========================================
avg_delay = total_delay / MAX_CUSTOMERS
avg_queue_length = area_queue / clock
server_utilization = area_server / clock
time_ended = clock

print("--- Simulation Results ---")
print(f"Mean Interarrival Time : {MEAN_INTERARRIVAL}")
print(f"Mean Service Time      : {MEAN_SERVICE}")
print(f"Number of Customers    : {MAX_CUSTOMERS}")
print("-" * 26)
print(f"Average Delay in Queue : {avg_delay:.4f} units")
print(f"Average Number in Queue: {avg_queue_length:.4f} customers")
print(f"Server Utilization     : {server_utilization:.4f} (Expected: {MEAN_SERVICE/MEAN_INTERARRIVAL:.4f})")
print(f"Time Simulation Ended  : {time_ended:.4f} units")


In [ ]:
# ==========================================
# 4. Visualization (Attractive Look)
# ==========================================
plt.figure(figsize=(12, 5))
# Plotting a subset of the simulation to see the queue behavior clearly
limit = min(500, len(time_points))
plt.step(time_points[:limit], queue_lengths[:limit], where='post', color='#1f77b4', linewidth=1.5)

plt.fill_between(time_points[:limit], queue_lengths[:limit], step='post', alpha=0.3, color='#1f77b4')
plt.title("Queue Length Over Time (First 500 events)", fontsize=14, fontweight='bold', color='#333333')
plt.xlabel("Simulation Time", fontsize=12)
plt.ylabel("Number of Customers in Queue", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.tight_layout()
plt.show()
